# Lecture 11: Transformer Architecture — Concepts — Answer Key
### NLP Course 2027 — Week 06

---
Complete answers for all four exercises in Lecture 11.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
print('Ready')

---
## Exercise 11.1 — Scaled Dot-Product Attention

**Task:** Implement `scaled_dot_product_attention` and plot attention weights.

**Formula:** $\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$

**Key concept:** Scaling by $\sqrt{d_k}$ prevents vanishing gradients from the softmax when $d_k$ is large (dot products grow in magnitude with dimension).

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (seq_q, d_k)  K: (seq_k, d_k)  V: (seq_k, d_v)
    Returns: output (seq_q, d_v), attention_weights (seq_q, seq_k)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)          # (seq_q, seq_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9) # fill masked positions with -inf
    attn_weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
    attn_weights /= attn_weights.sum(axis=-1, keepdims=True)  # softmax
    output = attn_weights @ V
    return output, attn_weights

np.random.seed(42)
tokens = ['cat', 'sat', 'mat']
seq_len, d_k, d_v = 3, 4, 4
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_v)

output, attn_weights = scaled_dot_product_attention(Q, K, V)
print('Attention weights (each row sums to 1):')
print(np.round(attn_weights, 4))
print(f'Row sums: {attn_weights.sum(axis=1)}')

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(attn_weights, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(tokens); ax.set_yticklabels(tokens)
ax.set_xlabel('Key'); ax.set_ylabel('Query')
ax.set_title('Scaled Dot-Product Attention Weights')
plt.colorbar(im, ax=ax)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{attn_weights[i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.tight_layout(); plt.show()

---
## Exercise 11.2 — Causal Multi-Head Attention

**Task:** Add a causal mask so each position can only attend to itself and earlier positions.

**Key concept:** Causal (autoregressive) masking is used in decoder-only models (GPT). It forces the model to predict token $t$ using only tokens $0..t-1$, preventing "cheating" by looking at future tokens.

In [ ]:
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.d_k = embed_dim // num_heads
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)

    def make_causal_mask(self, seq_len, device):
        """Lower-triangular mask: position i can attend to j <= i."""
        # True where attention is ALLOWED, False where masked
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).bool()
        return mask  # (seq_len, seq_len)

    def forward(self, x):
        B, S, _ = x.shape
        mask = self.make_causal_mask(S, x.device)  # (S, S)

        Q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)  # (B, H, S, dk)
        K = self.W_k(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)  # (B, H, S, S)
        scores = scores.masked_fill(~mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn   = torch.softmax(scores, dim=-1)
        out    = (attn @ V).transpose(1, 2).contiguous().view(B, S, self.embed_dim)
        return self.W_o(out)

causal_attn = CausalMultiHeadAttention(embed_dim=64, num_heads=4)
x   = torch.randn(2, 5, 64)
out = causal_attn(x)
print(f'Output shape: {out.shape}')  # (2, 5, 64)

# Visualize the causal mask
mask = causal_attn.make_causal_mask(5, 'cpu').numpy().astype(float)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(mask, cmap='Blues')
ax.set_title('Causal Mask (1=attend, 0=masked)')
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
plt.tight_layout(); plt.show()

---
## Exercise 11.3 — Learned Positional Encoding

**Task:** Implement `LearnedPositionalEncoding`. Compare with sinusoidal.

**Key concept:** Sinusoidal PE is deterministic (fixed math formula). Learned PE uses a trainable embedding table — the model discovers the best encoding for the task. BERT uses learned PE; the original Transformer used sinusoidal.

In [ ]:
class LearnedPositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)  # (1, seq_len)
        return x + self.pe(positions)


def sinusoidal_pe(max_len, d_model):
    pe = np.zeros((max_len, d_model))
    pos = np.arange(max_len)[:, None]
    div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(pos * div)
    pe[:, 1::2] = np.cos(pos * div[:d_model//2])
    return pe

# Test
learned_pe = LearnedPositionalEncoding(max_len=50, d_model=64)
x = torch.zeros(1, 50, 64)
print(f'LearnedPE output shape: {learned_pe(x).shape}')

# Compare visually
sin_pe = sinusoidal_pe(50, 64)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(sin_pe, aspect='auto', cmap='RdBu')
axes[0].set_title('Sinusoidal PE (deterministic)'); axes[0].set_xlabel('Dimension'); axes[0].set_ylabel('Position')
learned_vals = learned_pe.pe.weight.detach().numpy()
axes[1].imshow(learned_vals, aspect='auto', cmap='RdBu')
axes[1].set_title('Learned PE (random init, before training)'); axes[1].set_xlabel('Dimension')
plt.tight_layout(); plt.show()
print('Note: Learned PE starts random; the sinusoidal pattern emerges during training.')

---
## Exercise 11.4 — BERT Parameter Count

**Task:** Count BERT-base parameters analytically. Verify with Hugging Face.

**Key concept:** Most BERT parameters are in the FFN layers (4 × d_model intermediate size), not the attention heads. Understanding parameter counts helps reason about memory requirements and model compression strategies.

In [ ]:
def count_bert_params(num_layers=12, d_model=768, num_heads=12,
                      vocab_size=30522, max_len=512, ffn_factor=4):
    d_ff = d_model * ffn_factor

    # Embeddings
    token_emb    = vocab_size * d_model
    pos_emb      = max_len   * d_model
    segment_emb  = 2         * d_model
    emb_layernorm = 2        * d_model   # gamma + beta

    # Per-layer
    attn_qkv     = 3 * (d_model * d_model + d_model)   # W_Q, W_K, W_V + biases
    attn_out     = d_model * d_model + d_model          # W_O + bias
    attn_ln      = 2 * d_model                          # LayerNorm
    ffn_w1       = d_model * d_ff + d_ff               # FC1
    ffn_w2       = d_ff * d_model + d_model            # FC2
    ffn_ln       = 2 * d_model                          # LayerNorm
    per_layer    = attn_qkv + attn_out + attn_ln + ffn_w1 + ffn_w2 + ffn_ln

    # Pooler (CLS -> dense)
    pooler = d_model * d_model + d_model

    total = (token_emb + pos_emb + segment_emb + emb_layernorm
             + num_layers * per_layer + pooler)

    print(f'Embedding:      {token_emb + pos_emb + segment_emb + emb_layernorm:>12,}')
    print(f'Per-layer:      {per_layer:>12,}  × {num_layers} layers')
    print(f'All layers:     {num_layers * per_layer:>12,}')
    print(f'Pooler:         {pooler:>12,}')
    print(f'─' * 30)
    print(f'Total estimate: {total:>12,}')
    return total

estimated = count_bert_params()
print(f'Known actual:   {110_000_000:>12,}')
print()

try:
    from transformers import AutoModel
    model = AutoModel.from_pretrained('bert-base-uncased')
    actual = sum(p.numel() for p in model.parameters())
    print(f'HF verified:    {actual:>12,}')
    print(f'Estimate error: {abs(estimated - actual) / actual * 100:.1f}%')
except Exception as e:
    print(f'(HF verification skipped: {e})')

---
## Summary

| Exercise | Key Concept |
|----------|-------------|
| 11.1 | Scaling by √d_k prevents vanishing gradients; attention rows sum to 1 (softmax) |
| 11.2 | Causal mask: lower-triangular — enables autoregressive generation (GPT style) |
| 11.3 | Learned PE: trainable table; sinusoidal: deterministic; BERT uses learned |
| 11.4 | Most params in FFN (d_model×4×d_model); embeddings and attention are smaller |

---
*NLP Course 2027 — Week 06*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**